In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
FOLDER = '/content/drive/MyDrive/CPI_Research'
os.makedirs(FOLDER, exist_ok=True)
print("✅ Drive mounted:", os.listdir(FOLDER))

Mounted at /content/drive
✅ Drive mounted: ['bangladesh_MASTER_dataset.csv', 'lasso_selected_features.csv', 'session1_orders.json', 'fig_session1_predictions.png', 'fig_session2_predictions.png', 'FINAL_scoreboard.csv', 'fig_scoreboard.png', 'fig_eda_series.png', 'fig_correlation.png', 'FINAL_forecast_2026_2027.csv', 'FINAL_forecast_chart.png', 'all_model_results.csv']


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools, warnings, json
warnings.filterwarnings('ignore')

from sklearn.linear_model import LassoCV, Lasso
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

SEED = 42
np.random.seed(SEED)
print("✅ Ready. Seed fixed at", SEED)

✅ Ready. Seed fixed at 42


In [3]:
df = pd.read_csv(f'{FOLDER}/bangladesh_MASTER_dataset.csv',
                 index_col=0, parse_dates=True)

# Derive inflation directly from CPI (305 points, back to 2001)
df['Inflation_from_CPI'] = (df['CPI'].pct_change(12) * 100)

# ── WINDOWS ──────────────────────────────────────────────
TRAIN_END  = '2020-12-01'
TEST_START = '2021-01-01'
TEST_END   = '2026-04-01'   # last month where ALL variables exist

y_full = df['CPI'].loc['2000-01-01':TEST_END]          # univariate target
y_train = y_full.loc[:TRAIN_END]
y_test  = y_full.loc[TEST_START:]

EXOG = ['ExchangeRate_BDT_USD','Forex_Reserves_USDmn','BroadMoney_BDTmn',
        'Brent_Oil_USD','Fed_Funds_Rate','Gold_Price_Index',
        'FAO_Food_Index','FAO_Cereals_Index',
        'COVID_dummy','UkraineWar_dummy','BD_Unrest_dummy']

print(f"Univariate  → train {len(y_train)} months | test {len(y_test)} months")
print(f"Test window covers the 2022–23 inflation crisis ✅")

Univariate  → train 252 months | test 64 months
Test window covers the 2022–23 inflation crisis ✅


In [4]:
ar = SARIMAX(y_train, order=(3,2,3), enforce_stationarity=False,
             enforce_invertibility=False).fit(disp=False)
arima_pred = ar.append(y_test, refit=False).get_prediction(start=y_test.index[0]).predicted_mean

sa = SARIMAX(y_train, order=(2,1,2), seasonal_order=(0,1,1,12),
             enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sa.append(y_test, refit=False).get_prediction(start=y_test.index[0]).predicted_mean
print(round(np.sqrt(((y_test-sarima_pred)**2).mean()),4))  # must print 1.3053

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used

1.3053


In [5]:
# =============================================================
# Q1 UPGRADE EXPERIMENTS — run in Google Colab (Session 4)
# Reproduces: naive benchmarks, DM tests, multi-horizon,
# inflation-rate metrics, 10-seed neural runs.
# Paste each CELL into your existing pipeline after loading df
# (bangladesh_MASTER_dataset.csv) exactly as in Sessions 1–3.
# =============================================================

# ---------- CELL A: setup (matches your Sessions 1–3) ----------
import numpy as np, pandas as pd, warnings, itertools, json
warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats

TRAIN_END, TEST_START, TEST_END = '2020-12-01', '2021-01-01', '2026-04-01'
y_full = df['CPI'].loc['2000-01-01':TEST_END]
y_train, y_test = y_full.loc[:TRAIN_END], y_full.loc[TEST_START:]

def metrics(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return dict(R2=round(r2_score(a,p),4), RMSE=round(np.sqrt(mean_squared_error(a,p)),4),
                MAE=round(mean_absolute_error(a,p),4),
                MAPE=round(np.mean(np.abs((a-p)/a))*100,3))

def dm_test(e1, e2, h=1):
    """Diebold–Mariano, squared loss, HLN small-sample correction.
    Negative DM => model 1 (first errors) more accurate."""
    e1, e2 = np.asarray(e1,float), np.asarray(e2,float)
    d = e1**2 - e2**2
    T = len(d); dbar = d.mean()
    lrv = np.mean((d-dbar)**2)
    for k in range(1, h):
        lrv += 2*np.mean((d[k:]-dbar)*(d[:-k]-dbar))
    dm = dbar/np.sqrt(lrv/T)
    hln = dm*np.sqrt((T+1-2*h+h*(h-1)/T)/T)
    return round(hln,3), round(2*(1-stats.t.cdf(abs(hln), df=T-1)),4)


In [6]:
# ---------- CELL B: naive benchmarks ----------
diffs = y_full.diff()
drift = diffs.expanding().mean().shift(1)          # causal expanding drift
naives = {
  'Naive (RW)':              y_full.shift(1).loc[TEST_START:],
  'RW + drift':              (y_full.shift(1) + drift).loc[TEST_START:],
  'Seasonal naive':          y_full.shift(12).loc[TEST_START:],
  'Seasonal naive + drift':  (y_full.shift(12) + 12*drift).loc[TEST_START:],
}
for name, p in naives.items():
    print(name, metrics(y_test, p))
    # optionally: log_result(name, y_test, p)

Naive (RW) {'R2': 0.9917, 'RMSE': np.float64(3.0882), 'MAE': 2.4653, 'MAPE': np.float64(1.009)}
RW + drift {'R2': 0.9933, 'RMSE': np.float64(2.7781), 'MAE': 2.1671, 'MAPE': np.float64(0.886)}
Seasonal naive {'R2': 0.6465, 'RMSE': np.float64(20.1382), 'MAE': 19.3546, 'MAPE': np.float64(7.8)}
Seasonal naive + drift {'R2': 0.8666, 'RMSE': np.float64(12.3732), 'MAE': 11.4028, 'MAPE': np.float64(4.527)}


In [7]:
infl_actual = (y_test / y_full.shift(12).loc[TEST_START:] - 1) * 100
for name, p in {'SARIMA': sarima_pred, 'ARIMA': arima_pred, **naives}.items():
    ip = (pd.Series(np.asarray(p,float), index=y_test.index[:len(p)])
          / y_full.shift(12).loc[TEST_START:] - 1) * 100
    idx = ip.dropna().index
    print(name, 'RMSE(pp)=', round(np.sqrt(mean_squared_error(infl_actual.loc[idx], ip.loc[idx])),4),
          'MAE(pp)=', round(mean_absolute_error(infl_actual.loc[idx], ip.loc[idx]),4))

SARIMA RMSE(pp)= 0.5865 MAE(pp)= 0.4181
ARIMA RMSE(pp)= 1.1795 MAE(pp)= 0.8548
Naive (RW) RMSE(pp)= 1.3621 MAE(pp)= 1.0969
RW + drift RMSE(pp)= 1.2283 MAE(pp)= 0.9631
Seasonal naive RMSE(pp)= 8.6878 MAE(pp)= 8.4912
Seasonal naive + drift RMSE(pp)= 5.274 MAE(pp)= 4.9413


In [8]:
horizons = [1,3,6,12]
ar_fit = SARIMAX(y_train, order=(3,2,3), enforce_stationarity=False,
                 enforce_invertibility=False).fit(disp=False)
sa_fit = SARIMAX(y_train, order=(2,1,2), seasonal_order=(0,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
mh = {h: {} for h in horizons}
for label, fitted in [('SARIMA', sa_fit), ('ARIMA', ar_fit)]:
    fc = {h: [] for h in horizons}; act = {h: [] for h in horizons}
    for origin in y_test.index:
        hist_end = y_full.index.get_loc(origin) - 1
        ext = fitted.append(y_full.iloc[len(y_train):hist_end+1], refit=False) \
              if hist_end+1 > len(y_train) else fitted
        f = ext.get_forecast(steps=max(horizons)).predicted_mean
        for h in horizons:
            t = hist_end + h
            if t < len(y_full):
                fc[h].append(f.iloc[h-1]); act[h].append(y_full.iloc[t])
    for h in horizons:
        mh[h][label] = round(np.sqrt(mean_squared_error(act[h], fc[h])),4)
for h in horizons:
    fc_rwd, fc_sn, act = [], [], []
    for origin in y_test.index:
        i = y_full.index.get_loc(origin) - 1; t = i + h
        if t >= len(y_full): continue
        act.append(y_full.iloc[t])
        src = t - 12
        fc_sn.append(y_full.iloc[src] if src <= i else y_full.iloc[i - ((i-src) % 12)])
        fc_rwd.append(y_full.iloc[i] + h*diffs.iloc[:i+1].mean())
    mh[h]['RW + drift'] = round(np.sqrt(mean_squared_error(act, fc_rwd)),4)
    mh[h]['Seasonal naive'] = round(np.sqrt(mean_squared_error(act, fc_sn)),4)
print(pd.DataFrame(mh))

                     1        3        6        12
SARIMA           1.3053   2.1830   3.5277   5.7111
ARIMA            2.6239   6.0416   8.1588  10.5352
RW + drift       2.7781   6.1261   8.8905  14.0192
Seasonal naive  20.1382  20.3869  20.7732  21.6124


In [9]:
import tensorflow as tf, random
from tensorflow import keras
from tensorflow.keras import layers
random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print("TF", tf.__version__)

TF 2.20.0


In [10]:
TRAIN_END, TEST_START, TEST_END = '2020-12-01', '2021-01-01', '2026-04-01'
LOOKBACK = 12

FEATS = ['CPI','ExchangeRate_BDT_USD','Forex_Reserves_USDmn','BroadMoney_BDTmn',
         'Brent_Oil_USD','Fed_Funds_Rate','Gold_Price_Index','FAO_Food_Index',
         'FAO_Cereals_Index','COVID_dummy','UkraineWar_dummy','BD_Unrest_dummy']

data = df.loc['2002-01-01':TEST_END, FEATS].dropna()
target_diff = data['CPI'].diff()          # predict the CHANGE
cpi_lag1    = data['CPI'].shift(1)        # to reconstruct levels later

scaler = StandardScaler().fit(data.loc[:TRAIN_END])     # TRAIN ONLY — no leakage
data_s = pd.DataFrame(scaler.transform(data), index=data.index, columns=FEATS)

X, y, y_dates, base = [], [], [], []
for i in range(LOOKBACK, len(data_s)):
    if pd.isna(target_diff.iloc[i]): continue
    X.append(data_s.iloc[i-LOOKBACK:i].values)
    y.append(target_diff.iloc[i])
    base.append(cpi_lag1.iloc[i])
    y_dates.append(data_s.index[i])

X, y, base = np.array(X), np.array(y), np.array(base)
y_dates = pd.DatetimeIndex(y_dates)

tr = y_dates <= TRAIN_END
Xtr, ytr = X[tr], y[tr]
Xte, yte, base_te = X[~tr], y[~tr], base[~tr]
dates_te = y_dates[~tr]
actual_te = base_te + yte                 # actual CPI levels for evaluation

print(f"✅ Train: {Xtr.shape} | Test: {Xte.shape} (12 months × {len(FEATS)} features each)")

✅ Train: (216, 12, 12) | Test: (64, 12, 12) (12 months × 12 features each)


In [11]:
keras.backend.clear_session()
tf.random.set_seed(SEED)

lstm = keras.Sequential([
    layers.Input(shape=(LOOKBACK, len(FEATS))),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])
lstm.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

hist = lstm.fit(Xtr, ytr, validation_split=0.15, epochs=300, batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)],
    verbose=0)
print(f"Stopped at epoch {len(hist.history['loss'])}")

lstm_pred = base_te + lstm.predict(Xte, verbose=0).ravel()   # change → level
print('LSTM', metrics(actual_te, lstm_pred))

Stopped at epoch 50
LSTM {'R2': 0.9937, 'RMSE': np.float64(2.6889), 'MAE': 2.0589, 'MAPE': np.float64(0.838)}


In [13]:
keras.backend.clear_session()
tf.random.set_seed(SEED)

inp = layers.Input(shape=(LOOKBACK, len(FEATS)))
x = layers.Dense(32)(inp)                                   # embed
pos = layers.Embedding(LOOKBACK, 32)(tf.range(LOOKBACK))    # positional encoding
x = x + pos
attn = layers.MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
x = layers.LayerNormalization()(x + attn)
ff = layers.Dense(64, activation='relu')(x); ff = layers.Dense(32)(ff)
x = layers.LayerNormalization()(x + ff)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(1)(x)

trans = keras.Model(inp, out)
trans.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

hist = trans.fit(Xtr, ytr, validation_split=0.15, epochs=300, batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)],
    verbose=0)
print(f"Stopped at epoch {len(hist.history['loss'])}")

trans_pred = base_te + trans.predict(Xte, verbose=0).ravel()
print('Transformer', metrics(actual_te, trans_pred))

Stopped at epoch 52
Transformer {'R2': 0.9936, 'RMSE': np.float64(2.72), 'MAE': 2.1329, 'MAPE': np.float64(0.876)}


In [14]:
# ---------- CELL C: DM battery vs SARIMA ----------
# Requires: sarima_pred, arima_pred, lstm_pred, trans_pred from Sessions 1–2,
# plus hybrid predictions from Session 3 (align indices where needed).
e_sarima = (y_test - sarima_pred).values
competitors = {'ARIMA': arima_pred, **naives}
try:
    competitors['LSTM'] = pd.Series(lstm_pred, index=dates_te)
    competitors['Transformer'] = pd.Series(trans_pred, index=dates_te)
except NameError:
    print('run Session 2 first for neural DM tests')
for name, p in competitors.items():
    idx = p.dropna().index.intersection(y_test.index)
    e2 = (y_test.loc[idx] - p.loc[idx]).values
    e1 = (y_test.loc[idx] - sarima_pred.loc[idx]).values
    print(f"SARIMA vs {name}: DM, p = {dm_test(e1, e2)}")

SARIMA vs ARIMA: DM, p = (np.float64(-3.294), np.float64(0.0016))
SARIMA vs Naive (RW): DM, p = (np.float64(-5.028), np.float64(0.0))
SARIMA vs RW + drift: DM, p = (np.float64(-4.829), np.float64(0.0))
SARIMA vs Seasonal naive: DM, p = (np.float64(-16.104), np.float64(0.0))
SARIMA vs Seasonal naive + drift: DM, p = (np.float64(-12.323), np.float64(0.0))
SARIMA vs LSTM: DM, p = (np.float64(-5.001), np.float64(0.0))
SARIMA vs Transformer: DM, p = (np.float64(-4.937), np.float64(0.0))


In [15]:
SEEDS = [42, 7, 13, 21, 99, 123, 256, 314, 777, 2024]
seed_rows = []

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

for s in SEEDS:
    keras.backend.clear_session(); set_seed(s)
    m = keras.Sequential([
        layers.Input(shape=(LOOKBACK, len(FEATS))),
        layers.LSTM(64), layers.Dropout(0.2),
        layers.Dense(32, activation='relu'), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    m.fit(Xtr, ytr, validation_split=0.15, epochs=300, batch_size=16,
          callbacks=[keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)],
          verbose=0)
    pred = base_te + m.predict(Xte, verbose=0).ravel()
    rmse = np.sqrt(mean_squared_error(actual_te, pred))
    seed_rows.append({'model':'LSTM','seed':s,'RMSE':round(rmse,4),
                      'MAE':round(mean_absolute_error(actual_te,pred),4)})
    print('LSTM', s, round(rmse,4))
pd.DataFrame(seed_rows).to_csv(f'{FOLDER}/multiseed_results.csv', index=False)

LSTM 42 2.6889


LSTM 7 2.6591


LSTM 13 2.804
LSTM 21 2.8605
LSTM 99 2.6855
LSTM 123 2.7952
LSTM 256 2.8032
LSTM 314 2.7991
LSTM 777 2.754
LSTM 2024 2.6576


In [16]:
# ---------- CELL 6b: Transformer, 10 seeds ----------
for s in SEEDS:
    keras.backend.clear_session(); set_seed(s)

    inp = layers.Input(shape=(LOOKBACK, len(FEATS)))
    x = layers.Dense(32)(inp)                                   # embed
    pos = layers.Embedding(LOOKBACK, 32)(tf.range(LOOKBACK))    # positional encoding
    x = x + pos
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
    x = layers.LayerNormalization()(x + attn)
    ff = layers.Dense(64, activation='relu')(x); ff = layers.Dense(32)(ff)
    x = layers.LayerNormalization()(x + ff)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1)(x)

    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    m.fit(Xtr, ytr, validation_split=0.15, epochs=300, batch_size=16,
          callbacks=[keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)],
          verbose=0)

    pred = base_te + m.predict(Xte, verbose=0).ravel()
    rmse = np.sqrt(mean_squared_error(actual_te, pred))
    seed_rows.append({'model': 'Transformer', 'seed': s, 'RMSE': round(rmse, 4),
                      'MAE': round(mean_absolute_error(actual_te, pred), 4)})
    print('Transformer', s, round(rmse, 4))

pd.DataFrame(seed_rows).to_csv(f'{FOLDER}/multiseed_results.csv', index=False)
print(f"✅ saved — {len(seed_rows)} rows so far")

Transformer 42 2.7119
Transformer 7 2.6391
Transformer 13 2.7009
Transformer 21 3.1229
Transformer 99 2.6725
Transformer 123 2.6981
Transformer 256 2.6978
Transformer 314 2.8576
Transformer 777 2.8867
Transformer 2024 2.7768
✅ saved — 20 rows so far


In [17]:
ext = sa.append(y_test, refit=False)
test_fit = ext.get_prediction(start=y_test.index[0]).predicted_mean
train_fit = sa.get_prediction(start=y_train.index[0]).predicted_mean
resid_train = (y_train - train_fit).dropna()
resid_test  = (y_test - test_fit)
sc = StandardScaler().fit(resid_train.values.reshape(-1,1))
r_all = pd.concat([resid_train, resid_test])
r_s = pd.Series(sc.transform(r_all.values.reshape(-1,1)).ravel(), index=r_all.index)
Xr, yr, dr = [], [], []
v = r_s.values
for i in range(LOOKBACK, len(v)):
    Xr.append(v[i-LOOKBACK:i]); yr.append(v[i]); dr.append(r_s.index[i])
Xr = np.array(Xr)[..., None]; yr = np.array(yr); dr = pd.DatetimeIndex(dr)
trh = dr <= TRAIN_END
e_sar = (y_test - sarima_pred)

for s in SEEDS:
    keras.backend.clear_session(); set_seed(s)
    m = keras.Sequential([layers.Input(shape=(LOOKBACK,1)), layers.LSTM(32),
                          layers.Dropout(0.2), layers.Dense(16, activation='relu'),
                          layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    m.fit(Xr[trh], yr[trh], validation_split=0.15, epochs=300, batch_size=16,
          callbacks=[keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)],
          verbose=0)
    r_pred = sc.inverse_transform(m.predict(Xr[~trh], verbose=0)).ravel()
    final = test_fit.loc[dr[~trh]] + r_pred
    a = y_test.loc[dr[~trh]]
    rmse = np.sqrt(mean_squared_error(a, final))
    dm, p = dm_test(e_sar.loc[dr[~trh]].values, (a - final).values)
    seed_rows.append({'model':'SARIMA-LSTM','seed':s,'RMSE':round(rmse,4),
                      'MAE':round(mean_absolute_error(a,final),4),'DM':dm,'p':p})
    print('SARIMA-LSTM', s, round(rmse,4), 'DM', dm, 'p', p)
pd.DataFrame(seed_rows).to_csv(f'{FOLDER}/multiseed_results.csv', index=False)

SARIMA-LSTM 42 1.303 DM 0.511 p 0.6112
SARIMA-LSTM 7 1.3111 DM -0.627 p 0.5332
SARIMA-LSTM 13 1.3664 DM -1.094 p 0.2781
SARIMA-LSTM 21 1.3026 DM 0.435 p 0.6649
SARIMA-LSTM 99 1.3084 DM -0.632 p 0.5299
SARIMA-LSTM 123 1.6792 DM -2.575 p 0.0124
SARIMA-LSTM 256 1.4052 DM -2.004 p 0.0494
SARIMA-LSTM 314 1.6598 DM -3.374 p 0.0013
SARIMA-LSTM 777 1.3579 DM -1.654 p 0.103
SARIMA-LSTM 2024 1.3056 DM -0.093 p 0.9264
